# CNN + MLP: Peptide (ESM-2) × Bacteria (Oligonucleotide) → MIC
**Google Colab notebook** — clones the project from GitHub and reads data directly from the repo.

### Architecture
- **Peptide branch**: ESM-2 embedding (480d) → reshape → 1D-CNN (multi-scale filters) → global pooling
- **Bacteria branch**: Oligonucleotide composition (1344d) → reshape → 1D-CNN → global pooling
- **Fusion**: concatenate branch outputs → MLP regression head

### Why CNN?
- Faster training than transformer (~10-20x)
- Multi-scale 1D convolutions capture local patterns in feature space
- Less prone to overfitting on this data size (~40K samples)


In [ ]:
#@title 1. Clone repo & install dependencies
import os

REPO_URL = "https://github.com/nmach22/amp-prediction.git"
PROJECT_DIR = "/content/amp-prediction"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)

!pip -q install "transformers>=4.40" pyarrow scikit-learn scipy seaborn 2>/dev/null

import torch, transformers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Working dir: {os.getcwd()}")


In [ ]:
#@title 2. Verify data files
from pathlib import Path

required_files = {
    "data/raw/amp_mic_activities.csv": "Raw MIC data",
    "data/processed/embeddings/genome/oligo_composition.parquet": "Bacteria genome vectors",
}

for fpath, desc in required_files.items():
    exists = Path(fpath).exists()
    status = "ok" if exists else "MISSING"
    print(f"  [{status}] {desc}: {fpath}")
    if not exists:
        raise FileNotFoundError(f"Required file missing: {fpath}")

esm_cache = Path("data/processed/embeddings/facebook_esm2_t12_35M_UR50D_mic_embeddings.npz")
HAS_ESM_CACHE = esm_cache.exists()
print(f"  ESM-2 cache: {HAS_ESM_CACHE}")


In [ ]:
#@title 3. Load & prepare data
import sys
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import re

df = pd.read_csv("data/raw/amp_mic_activities.csv")
print(f"Raw rows: {len(df):,}")

df = df.dropna(subset=["sequence", "target_activity_name", "activity"])
df["sequence"] = df["sequence"].str.upper().str.strip()
df["target_activity_name"] = df["target_activity_name"].str.strip()
df["activity"] = pd.to_numeric(df["activity"], errors="coerce")
df = df[df["activity"] > 0]
df = df[df["sequence"].str.len() > 0]
df = df[~df["sequence"].str.contains(r"[^ACDEFGHIKLMNPQRSTVWY]", flags=re.IGNORECASE)]
df["log_mic"] = np.log10(df["activity"])
df["species"] = df["target_activity_name"].apply(lambda x: " ".join(str(x).split()[:2]))

agg = df.groupby(["sequence", "species"], as_index=False).agg(
    log_mic=("log_mic", "mean"),
    n_measurements=("log_mic", "size"),
    target_activity_name=("target_activity_name", "first")
)
print(f"After aggregation: {len(agg):,} unique (peptide, species) pairs")
print(f"Unique peptides: {agg['sequence'].nunique():,}")
print(f"Unique species: {agg['species'].nunique():,}")


In [ ]:
#@title 4. Load oligonucleotide genome features
from src.features.genome import GenomeEncoder

genome_enc = GenomeEncoder()
oligo = pd.read_parquet("data/processed/embeddings/genome/oligo_composition.parquet")
print(f"Oligo features: {oligo.shape[0]} species x {oligo.shape[1]-1} features")

oligo_cols = [c for c in oligo.columns if c != "species"]
agg = agg.merge(oligo, on="species", how="left")

matched = agg[oligo_cols[0]].notna().sum()
print(f"Genome match rate: {matched}/{len(agg)} ({matched/len(agg)*100:.1f}%)")
agg[oligo_cols] = agg[oligo_cols].fillna(0.0)


In [ ]:
#@title 5. Load or compute ESM-2 peptide embeddings
from pathlib import Path

ESM2_MODEL = "facebook/esm2_t12_35M_UR50D"
ESM2_DIM = 480
CACHE_PATH = Path("data/processed/embeddings/facebook_esm2_t12_35M_UR50D_mic_embeddings.npz")

unique_seqs = sorted(agg["sequence"].unique())
print(f"Need embeddings for {len(unique_seqs)} unique sequences")

seq_to_emb = {}
if CACHE_PATH.exists():
    cache = np.load(CACHE_PATH, allow_pickle=True)
    for s, e in zip(cache["sequences"], cache["embeddings"]):
        seq_to_emb[s] = e
    print(f"Loaded {len(seq_to_emb)} embeddings from cache")

missing_seqs = [s for s in unique_seqs if s not in seq_to_emb]
print(f"Missing sequences: {len(missing_seqs)}")

if missing_seqs:
    from transformers import AutoTokenizer, AutoModel
    print(f"Computing ESM-2 embeddings for {len(missing_seqs)} sequences on {DEVICE}...")

    tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL)
    esm_model = AutoModel.from_pretrained(ESM2_MODEL).eval().to(DEVICE)

    batch_size = 32
    with torch.no_grad():
        for i in range(0, len(missing_seqs), batch_size):
            batch = missing_seqs[i:i+batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=512,
                return_special_tokens_mask=True
            ).to(DEVICE)
            outputs = esm_model(**inputs)
            mask = inputs["attention_mask"].bool() & ~inputs["special_tokens_mask"].bool()
            mask_3d = mask.unsqueeze(-1).float()
            pooled = (outputs.last_hidden_state * mask_3d).sum(1) / mask_3d.sum(1).clamp(min=1)
            for seq, emb in zip(batch, pooled.cpu().numpy()):
                seq_to_emb[seq] = emb
            if (i // batch_size) % 50 == 0:
                print(f"  [{i+len(batch)}/{len(missing_seqs)}]")

    del esm_model
    torch.cuda.empty_cache()

    all_seqs = sorted(seq_to_emb.keys())
    np.savez(CACHE_PATH, sequences=all_seqs, embeddings=np.array([seq_to_emb[s] for s in all_seqs]))
    print(f"Updated cache: {len(all_seqs)} sequences")

peptide_embeddings = np.vstack([seq_to_emb[s] for s in agg["sequence"]])
print(f"Peptide embeddings: {peptide_embeddings.shape}")


In [ ]:
#@title 6. Train/validation split (grouped by peptide)
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(agg, groups=agg["sequence"]))

print(f"Train: {len(train_idx):,} | Val: {len(val_idx):,}")

genome_features = agg[oligo_cols].values.astype(np.float32)

X_pep_train = peptide_embeddings[train_idx]
X_pep_val = peptide_embeddings[val_idx]
X_gen_train = genome_features[train_idx]
X_gen_val = genome_features[val_idx]
y_train = agg["log_mic"].values[train_idx].astype(np.float32)
y_val = agg["log_mic"].values[val_idx].astype(np.float32)

print(f"Feature dims: peptide={X_pep_train.shape[1]}, genome={X_gen_train.shape[1]}")


In [ ]:
#@title 7. Normalize features
from sklearn.preprocessing import StandardScaler

pep_scaler = StandardScaler()
gen_scaler = StandardScaler()

X_pep_train = pep_scaler.fit_transform(X_pep_train)
X_pep_val = pep_scaler.transform(X_pep_val)
X_gen_train = gen_scaler.fit_transform(X_gen_train)
X_gen_val = gen_scaler.transform(X_gen_val)

print(f"Peptide features normalized: mean={X_pep_train.mean():.4f}, std={X_pep_train.std():.4f}")
print(f"Genome features normalized: mean={X_gen_train.mean():.4f}, std={X_gen_train.std():.4f}")


In [ ]:
#@title 8. Define Multi-Scale CNN + MLP Model
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiScaleConv1DBlock(nn.Module):
    """Parallel conv1d with different kernel sizes, concatenated."""

    def __init__(self, in_channels, out_channels_per_scale, kernel_sizes=(3, 5, 7), dropout=0.1):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels, out_channels_per_scale, k, padding=k // 2),
                nn.BatchNorm1d(out_channels_per_scale),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            for k in kernel_sizes
        ])

    def forward(self, x):
        return torch.cat([conv(x) for conv in self.convs], dim=1)


class ResidualConvBlock(nn.Module):
    """Conv block with residual connection."""

    def __init__(self, channels, kernel_size=3, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.conv(x))


class ConvBranch(nn.Module):
    """1D CNN branch: flat vector -> sequence -> multi-scale conv -> residual blocks -> pool."""

    def __init__(self, input_dim, n_tokens, hidden_channels=128,
                 n_res_blocks=2, kernel_sizes=(3, 5, 7), dropout=0.15):
        super().__init__()
        self.n_tokens = n_tokens
        token_dim = (input_dim + n_tokens - 1) // n_tokens  # ceil division
        self.proj_dim = token_dim * n_tokens

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, self.proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Multi-scale conv: token_dim -> hidden_channels * n_scales
        n_scales = len(kernel_sizes)
        out_per_scale = hidden_channels // n_scales
        self.multi_scale = MultiScaleConv1DBlock(token_dim, out_per_scale, kernel_sizes, dropout)
        total_ch = out_per_scale * n_scales

        # Channel projection to hidden_channels
        self.channel_proj = nn.Sequential(
            nn.Conv1d(total_ch, hidden_channels, 1),
            nn.BatchNorm1d(hidden_channels),
            nn.GELU(),
        )

        # Residual conv blocks
        self.res_blocks = nn.Sequential(*[
            ResidualConvBlock(hidden_channels, 3, dropout)
            for _ in range(n_res_blocks)
        ])

        # Second multi-scale layer (deeper features)
        out_per_scale2 = hidden_channels // n_scales
        self.multi_scale2 = MultiScaleConv1DBlock(hidden_channels, out_per_scale2, kernel_sizes, dropout)
        self.output_channels = out_per_scale2 * n_scales

    def forward(self, x):
        B = x.size(0)
        x = self.input_proj(x)
        x = x.view(B, self.n_tokens, -1).permute(0, 2, 1)  # (B, token_dim, n_tokens)

        x = self.multi_scale(x)      # (B, hidden_ch, n_tokens)
        x = self.channel_proj(x)     # (B, hidden_channels, n_tokens)
        x = self.res_blocks(x)       # (B, hidden_channels, n_tokens)
        x = self.multi_scale2(x)     # (B, output_channels, n_tokens)

        # Global pooling: max + avg
        x_max = F.adaptive_max_pool1d(x, 1).squeeze(-1)
        x_avg = F.adaptive_avg_pool1d(x, 1).squeeze(-1)
        return torch.cat([x_max, x_avg], dim=1)  # (B, output_channels * 2)


class CNNMicRegressor(nn.Module):
    """
    Two-branch CNN for peptide x bacteria MIC prediction.
    Each branch: features -> reshape -> multi-scale CNN + residual blocks -> pool
    Fusion: concatenate -> deep MLP head
    """

    def __init__(
        self,
        peptide_dim=480,
        genome_dim=1344,
        hidden_channels=128,
        n_res_blocks=2,
        kernel_sizes=(3, 5, 7),
        mlp_hidden=(512, 256, 128),
        dropout=0.15,
        n_tokens_peptide=10,
        n_tokens_genome=16,
    ):
        super().__init__()

        self.pep_branch = ConvBranch(
            peptide_dim, n_tokens_peptide, hidden_channels,
            n_res_blocks, kernel_sizes, dropout
        )
        self.gen_branch = ConvBranch(
            genome_dim, n_tokens_genome, hidden_channels,
            n_res_blocks, kernel_sizes, dropout
        )

        # MLP head
        fused_dim = self.pep_branch.output_channels * 2 + self.gen_branch.output_channels * 2
        layers = []
        in_dim = fused_dim
        for h in mlp_hidden:
            layers.extend([
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.head = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, peptide_features, genome_features):
        pep = self.pep_branch(peptide_features)
        gen = self.gen_branch(genome_features)
        fused = torch.cat([pep, gen], dim=1)
        return self.head(fused)


# Instantiate
model = CNNMicRegressor(
    peptide_dim=X_pep_train.shape[1],
    genome_dim=X_gen_train.shape[1],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Architecture: 2-branch CNN (multi-scale k=3,5,7 + residual) + MLP head")
print(f"Input: peptide({X_pep_train.shape[1]}d) + genome({X_gen_train.shape[1]}d)")


In [ ]:
#@title 9. PyTorch Dataset & DataLoaders

class MICDataset(torch.utils.data.Dataset):
    def __init__(self, pep, gen, targets):
        self.pep = torch.tensor(pep, dtype=torch.float32)
        self.gen = torch.tensor(gen, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.pep[idx], self.gen[idx], self.targets[idx]


BATCH_SIZE = 512
train_dataset = MICDataset(X_pep_train, X_gen_train, y_train)
val_dataset = MICDataset(X_pep_val, X_gen_val, y_val)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=1024, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
#@title 10. Training loop
import time
from collections import defaultdict

MAX_EPOCHS = 300
PATIENCE = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
)
loss_fn = nn.HuberLoss(delta=1.0)

history = defaultdict(list)
best_val_mae = float("inf")
best_state = None
epochs_no_improve = 0

print(f"Training for up to {MAX_EPOCHS} epochs (patience={PATIENCE})")
print(f"Batch size: {BATCH_SIZE} | LR: {LR}")
print("-" * 70)

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    # --- Train ---
    model.train()
    for pep, gen, y in train_loader:
        pep, gen, y = pep.to(DEVICE), gen.to(DEVICE), y.to(DEVICE)
        pred = model(pep, gen)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    scheduler.step()

    # --- Validate ---
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for pep, gen, y in val_loader:
            pep, gen = pep.to(DEVICE), gen.to(DEVICE)
            val_preds.append(model(pep, gen).cpu())
            val_targets.append(y)

    val_preds = torch.cat(val_preds).numpy().ravel()
    val_targets = torch.cat(val_targets).numpy().ravel()
    val_mae = np.mean(np.abs(val_preds - val_targets))
    val_rmse = np.sqrt(np.mean((val_preds - val_targets)**2))
    ss_res = np.sum((val_targets - val_preds)**2)
    ss_tot = np.sum((val_targets - val_targets.mean())**2)
    val_r2 = 1 - ss_res / ss_tot

    # Train MAE (clean eval)
    train_preds = []
    with torch.no_grad():
        for pep, gen, y in train_loader:
            pep, gen = pep.to(DEVICE), gen.to(DEVICE)
            train_preds.append(model(pep, gen).cpu())
    train_preds = torch.cat(train_preds).numpy().ravel()
    train_mae = np.mean(np.abs(train_preds - y_train))

    elapsed = time.time() - t0
    history["train_mae"].append(train_mae)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)
    history["val_r2"].append(val_r2)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    if val_mae < best_val_mae - 1e-4:
        best_val_mae = val_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        marker = " *"
    else:
        epochs_no_improve += 1
        marker = ""

    if epoch % 5 == 0 or epochs_no_improve == 0:
        print(f"Epoch {epoch:3d} | train={train_mae:.4f} | "
              f"val={val_mae:.4f} | r2={val_r2:.4f} | "
              f"lr={optimizer.param_groups[0]['lr']:.2e} | {elapsed:.1f}s{marker}")

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (best val_mae={best_val_mae:.4f})")
        break

if best_state:
    model.load_state_dict(best_state)
    model.to(DEVICE)
print(f"\nBest model: val_mae={best_val_mae:.4f}")


In [ ]:
#@title 11. Final evaluation & plots
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    final_preds = []
    for pep, gen, y in val_loader:
        pep, gen = pep.to(DEVICE), gen.to(DEVICE)
        final_preds.append(model(pep, gen).cpu())
    final_preds = torch.cat(final_preds).numpy().ravel()

final_mae = np.mean(np.abs(final_preds - y_val))
final_rmse = np.sqrt(np.mean((final_preds - y_val)**2))
ss_res = np.sum((y_val - final_preds)**2)
ss_tot = np.sum((y_val - y_val.mean())**2)
final_r2 = 1 - ss_res / ss_tot

print("=" * 50)
print("FINAL VALIDATION METRICS (CNN + MLP)")
print("=" * 50)
print(f"  MAE:  {final_mae:.4f}")
print(f"  RMSE: {final_rmse:.4f}")
print(f"  R2:   {final_r2:.4f}")
print("=" * 50)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(history["train_mae"], label="Train MAE", alpha=0.8)
axes[0].plot(history["val_mae"], label="Val MAE", alpha=0.8)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MAE")
axes[0].set_title("Learning Curves"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_val, final_preds, alpha=0.1, s=5)
lims = [min(y_val.min(), final_preds.min()), max(y_val.max(), final_preds.max())]
axes[1].plot(lims, lims, "r--", linewidth=1)
axes[1].set_xlabel("Actual log10(MIC)"); axes[1].set_ylabel("Predicted")
axes[1].set_title(f"Pred vs Actual (R2={final_r2:.3f})"); axes[1].grid(True, alpha=0.3)

residuals = final_preds - y_val
axes[2].hist(residuals, bins=50, edgecolor="white", alpha=0.7)
axes[2].axvline(0, color="red", linestyle="--")
axes[2].set_xlabel("Residual"); axes[2].set_ylabel("Count")
axes[2].set_title(f"Residuals (MAE={final_mae:.3f})"); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
Path("results/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("results/figures/genome_cnn_mic.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
#@title 12. Save model checkpoint
from pathlib import Path

Path("results/checkpoints").mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "pep_scaler_mean": pep_scaler.mean_,
    "pep_scaler_scale": pep_scaler.scale_,
    "gen_scaler_mean": gen_scaler.mean_,
    "gen_scaler_scale": gen_scaler.scale_,
    "config": {
        "peptide_dim": X_pep_train.shape[1],
        "genome_dim": X_gen_train.shape[1],
        "hidden_channels": 128,
        "n_res_blocks": 2,
        "kernel_sizes": (3, 5, 7),
        "mlp_hidden": (512, 256, 128),
        "dropout": 0.15,
    },
    "metrics": {
        "best_val_mae": best_val_mae,
        "final_mae": final_mae,
        "final_rmse": final_rmse,
        "final_r2": final_r2,
    },
    "history": dict(history),
}

ckpt_path = "results/checkpoints/genome_cnn_mic.pt"
torch.save(checkpoint, ckpt_path)
print(f"Saved: {ckpt_path}")

from google.colab import files
files.download(ckpt_path)
files.download("results/figures/genome_cnn_mic.png")
